# Variational Auto-Encoder (VAE)

本 Notebook 實作一個 VAE，以 MNIST 手寫數字資料集進行訓練，並透過 TensorBoard 可視化訓練過程。

**架構概覽**
- **Encoder**：CNN → 輸出 μ 和 log σ²
- **Reparameterize**：z = μ + σ × ε
- **Decoder**：反卷積網路，重建原始影像
- **Loss**：重構損失 (BCE) + KL Divergence

## 1. 安裝必要套件

In [ ]:
!pip install torchinfo
!pip install tensorboard

import os
import numpy as np
from tqdm import tqdm
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision
from torchinfo import summary
import matplotlib.pyplot as plt
from torchvision import transforms
from torchvision.utils import make_grid
from torch.utils.data import DataLoader
from torch.utils.tensorboard import SummaryWriter


## 1. Variational Auto-Encoder網路架構搭建

分別定義 `Encoder`、`Decoder` 與串接兩者的 `VAE` 類別。

- **Encoder**：多層 Conv2d → 輸出 `μ` 與 `log_var`
- **Decoder**：全連接層 + ConvTranspose2d 重建影像
- **reparameterize**：將 `log_var` 轉為 `std`，並加入隨機雜訊 `ε`（訓練時），推論時 `ε = 0`

In [ ]:

class Encoder(nn.Module):
    def __init__(self, z_dim):
        super(Encoder, self).__init__()
        self.model = nn.Sequential(
            nn.Conv2d(1, 32, 5, 1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.Conv2d(32, 64, 3, 2),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.Conv2d(64, 64, 3, 2),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.Conv2d(64, 64, 3, 2),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.Flatten(),
            nn.Linear(64 * 2 * 2, 512),
            nn.BatchNorm1d(512),
            nn.ReLU()
        )
        self.fc_mu = nn.Linear(512, z_dim)
        self.fc_log_var = nn.Linear(512, z_dim)

    def forward(self, x):
        x = self.model(x)
        mu = self.fc_mu(x)
        log_var = self.fc_log_var(x)
        return mu, log_var


class Decoder(nn.Module):
    def __init__(self, z_dim):
        super(Decoder, self).__init__()
        self.fc = nn.Linear(z_dim, 7 * 7 * 128)
        self.conv1 = nn.ConvTranspose2d(128, 64, 4, 2, 1)
        self.conv2 = nn.ConvTranspose2d(64, 32, 4, 2, 1)
        self.conv3 = nn.Conv2d(32, 1, 3, 1, 1)

    def forward(self, z):
        x = self.fc(z)
        x = x.view(-1, 128, 7, 7)
        x = F.relu(self.conv1(x))
        x = F.relu(self.conv2(x))
        x = self.conv3(x)
        return x


class VAE(nn.Module):
    def __init__(self, z_dim):
        super(VAE, self).__init__()
        self.encoder = Encoder(z_dim)
        self.decoder = Decoder(z_dim)

    def reparameterize(self, mu, log_var):
        # 透過 mu, log_var 來生成樣本點 z
        std = torch.exp(0.5 * log_var)
        # 在訓練時, 加入雜訊 epsilon 來增加模型的 robustness
        if not self.training:
            eps = torch.zeros_like(std)
        else:
            eps = torch.randn_like(std)
        z = mu + eps * std
        return z

    def forward(self, x):
        mu, log_var = self.encoder(x)
        z = self.reparameterize(mu, log_var)
        x_hat = self.decoder(z)
        return x_hat, mu, log_var


印出模型架構-encoder

In [ ]:

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'使用裝置：{device}')

print('\n--- Encoder ---')
encoder = Encoder(z_dim=2)
summary(encoder, input_size=(1, 1, 28, 28))

印出模型架構-decoder

In [ ]:
print('\n--- Decoder ---')
decoder = Decoder(z_dim=2)
summary(decoder, input_size=(1, 2))

## 2. 輔助函式與 TensorBoard Callback 

- `denorm`：將正規化後的影像還原
- `reconstruction_loss`：BCE 重構損失
- `kl_divergence`：KL 散度損失，使 latent space 趨近標準常態分布
- `TensorBoardCallback`：每隔固定 epoch 將重構影像、latent space 分布及 Decoder 對潛空間的插值結果寫入 TensorBoard

In [ ]:

def denorm(x, mean=0.1307, std=0.3081):
    """將 Normalize 後的影像還原至 [0, 1] 範圍"""
    return x * std + mean


def reconstruction_loss(generated_img, target_img):
    """Binary Cross Entropy 重構損失"""
    return F.binary_cross_entropy_with_logits(generated_img, target_img, reduction='sum')


def kl_divergence(mu, log_var):
    """KL Divergence：使 latent distribution 趨近 N(0,1)"""
    return -0.5 * torch.mean(1 + log_var - mu.pow(2) - log_var.exp())


class TensorBoardCallback:
    def __init__(self, log_dir, z_dim, device=None):
        self.log_dir = log_dir
        self.writer = SummaryWriter(log_dir=log_dir)
        self.z_dim = z_dim
        self.device = device

    def __call__(self, model: VAE, images, labels, loss, step):
        # 紀錄 loss
        r_loss, kl_loss = loss
        self.writer.add_scalar('r_loss', r_loss.item(), step)
        self.writer.add_scalar('kl_loss', kl_loss.item(), step)

        # 記錄模型參數直方圖
        for name, param in model.named_parameters():
            self.writer.add_histogram(name, param, step)

        with torch.no_grad():
            model.eval()

            # 可視化重構結果（前 30 張）
            n = 30
            imgs = images[:n]
            recon_images, _, _ = model(imgs)
            input_grid = make_grid(denorm(imgs), nrow=5, normalize=True)
            output_grid = make_grid(nn.functional.sigmoid(recon_images), nrow=5, normalize=True)
            self.writer.add_image('input_images', input_grid, step)
            self.writer.add_image('output_images', output_grid, step)

            # 可視化 latent space 分布
            mu, log_var = model.encoder(imgs)
            encoded_z = model.reparameterize(mu, log_var)
            self.writer.add_histogram('mu', mu, step)
            self.writer.add_histogram('lv', log_var, step)
            self.writer.add_histogram('latent_space/encoded', encoded_z, step)

            # 在 latent space 均勻插值並可視化 Decoder 輸出
            v_std = int(encoded_z.std())
            v_mean = int(encoded_z.mean())
            x_pts = np.linspace(v_mean - v_std, v_mean + v_std, n)
            y_pts = np.linspace(v_mean - v_std, v_mean + v_std, n)
            z_grid = np.zeros((n, n, self.z_dim))
            for i, xi in enumerate(x_pts):
                for j, yj in enumerate(y_pts):
                    z_grid[i, j, 0] = xi
                    z_grid[i, j, 1] = yj
            z_tensor = torch.from_numpy(z_grid.reshape(n * n, self.z_dim)).float().to(self.device)
            self.writer.add_histogram('latent_space/z', z_tensor, step)
            recon_x = model.decoder(z_tensor)
            image_grid = make_grid(nn.functional.sigmoid(recon_x), nrow=n, normalize=True)
            self.writer.add_image('test_decoder', image_grid, step)


## 3. 啟動 TensorBoard


In [ ]:
%load_ext tensorboard
%tensorboard --logdir ./runs

## 4. 訓練 VAE 

- 資料集：MNIST
- 優化器：AdamW，lr = 1e-3
- 訓練輪數：200 epochs
- 每 10 個 epoch 呼叫 `TensorBoardCallback` 記錄訓練狀態

In [ ]:
def main():
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    print(f'使用裝置：{device}')

    # 資料前處理
    transform = transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize((0.1307,), (0.3081,))
    ])

    train_dataset = torchvision.datasets.MNIST(
        root='./data', train=True, transform=transform, download=True
    )
    # Colab 建議 num_workers=2
    train_loader = DataLoader(train_dataset, batch_size=128, shuffle=True, num_workers=2, pin_memory=True)

    # 初始化模型與優化器
    model = VAE(z_dim=2).to(device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3)

    # 初始化 TensorBoardCallback
    log_dir = './runs'
    callback = TensorBoardCallback(log_dir, z_dim=2, device=device)

    # 將模型架構寫入 TensorBoard
    model.eval()
    dummy_input = torch.rand(1, 1, 28, 28).to(device)
    callback.writer.add_graph(model, dummy_input)

    # 訓練迴圈
    num_epochs = 200
    loss_r, loss_kl = 0, 0

    for epoch in range(num_epochs):
        tqdm_loader = tqdm(train_loader, desc=f'Epoch {epoch + 1}/{num_epochs}', leave=False)
        if epoch > 0:
            tqdm_loader.set_postfix({'r_loss': f'{loss_r:.2f}', 'kl_loss': f'{loss_kl:.4f}'})

        images, batch_reconstruct, batch_kl = None, None, None

        for images, _ in tqdm_loader:
            model.train()
            images = images.to(device)
            optimizer.zero_grad()

            x_hat, mu, log_var = model(images)

            # 計算重構損失前先將影像 denormalize
            img_label = denorm(images)
            batch_reconstruct = reconstruction_loss(x_hat, img_label)
            batch_kl = kl_divergence(mu, log_var)
            loss = batch_reconstruct + batch_kl

            loss.backward()
            optimizer.step()

        if epoch % 10 == 0:
            callback(model, images, None, [batch_reconstruct, batch_kl], epoch)
            loss_r = batch_reconstruct.item()
            loss_kl = batch_kl.item()
            print(f'Epoch [{epoch}/{num_epochs}]  r_loss: {loss_r:.2f}  kl_loss: {loss_kl:.4f}')

    return model


trained_model = main()

## 5. 生成影像

從標準常態分布中直接採樣潛在向量 `z`，輸入 Decoder 生成新的手寫數字影像。

In [ ]:
trained_model.eval()

n_samples = 16
with torch.no_grad():
    z = torch.randn(n_samples, 2).to(device)   # 從 N(0,1) 採樣
    generated = trained_model.decoder(z)
    generated = torch.sigmoid(generated).cpu()

fig, axes = plt.subplots(2, 8, figsize=(16, 4))
for i, ax in enumerate(axes.flat):
    ax.imshow(generated[i].squeeze(), cmap='gray')
    ax.axis('off')
plt.tight_layout()
plt.show()

## 6. 潛空間插值視覺化

在二維潛空間中均勻取樣，觀察 Decoder 對不同 z 值的輸出連續性。

In [ ]:
n = 15  # 每軸採樣點數
grid_range = 2.0  # 採樣範圍 [-2, 2]

figure = np.zeros((28 * n, 28 * n))
grid_x = np.linspace(-grid_range, grid_range, n)
grid_y = np.linspace(-grid_range, grid_range, n)[::-1]

trained_model.eval()
with torch.no_grad():
    for i, yi in enumerate(grid_y):
        for j, xi in enumerate(grid_x):
            z_sample = torch.tensor([[xi, yi]], dtype=torch.float32).to(device)
            x_decoded = trained_model.decoder(z_sample)
            digit = torch.sigmoid(x_decoded).cpu().squeeze().numpy()
            figure[i * 28:(i + 1) * 28, j * 28:(j + 1) * 28] = digit

plt.figure(figsize=(10, 10))
plt.imshow(figure, cmap='gray')
plt.xlabel('z[0]')
plt.ylabel('z[1]')
plt.xticks(np.arange(0, 28 * n, 28) + 14, [f'{v:.1f}' for v in grid_x], rotation=45, fontsize=7)
plt.yticks(np.arange(0, 28 * n, 28) + 14, [f'{v:.1f}' for v in grid_y], fontsize=7)
plt.tight_layout()
plt.show()